# Toxic Gaming Chat Classifier - Colab Training

**By Alex You**

This notebook uses Colab as a GPU runtime while keeping the project code in Python modules.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU`, then run this cell.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

CUDA available: True
GPU: Tesla T4
Fri Jul 17 01:52:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

## 2. Mount Google Drive

Set `PROJECT_DIR` to the folder that contains this repository in Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Update this if your repo is stored somewhere else in Drive.
PROJECT_DIR = '/content/drive/MyDrive/APS360/Final_Project'
%cd $PROJECT_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/APS360/Final_Project


## 3. Install Dependencies

In [3]:
# %pip install -r requirements.txt
%pip install -e .


## 4. Verify Or Prepare Expert-Labelled Data

Download L2DTnH's `2_16000_chatlogs_english_only.csv` to `data/l2dtnh/l2dtnh_english.csv`. The preparation script normalizes the messages, removes contradictory normalized labels, and creates match-grouped train/validation/test splits.

In [4]:
from pathlib import Path
import subprocess

import pandas as pd

raw_path = Path('data/l2dtnh/l2dtnh_english.csv')
prepared_path = Path('data/l2dtnh/l2dtnh_prepared.csv')

if not raw_path.exists():
    raise FileNotFoundError(
        'Upload 2_16000_chatlogs_english_only.csv as data/l2dtnh/l2dtnh_english.csv.'
    )

# Always rebuild so grouped split assignments and the audit match the current code.
subprocess.run(['python', 'scripts/prepare_l2dtnh.py'], check=True)

df = pd.read_csv(prepared_path)
print(df.head())
print(pd.crosstab(df['split'], df['label']))
print('Groups per split:', df.groupby('split')['group_id'].nunique().to_dict())

                                          raw_text  \
0                            olaf is bronze hahaha   
1  yes i know kayle is too but u gyus are premades   
2                                            kayle   
3                                if u dont die top   
4                    kayle plays like a challenger   

                                              text  label  group_id  \
0                            olaf is bronze hahaha      1      1009   
1  yes i know kayle is too but u gyus are premades      1      1009   
2                                            kayle      0      1009   
3                                if u dont die top      0      1009   
4                    kayle plays like a challenger      0      1009   

   msg_index split  
0          0  test  
1          1  test  
2          2  test  
3          3  test  
4          4  test  
label     0    1
split           
test   1939  195
train  9123  927
val    1911  198
Groups per split: {'test': 16, 'train

## 5. Context-Window + Hybrid Screen (Validation Only, Then Frozen Test)

Rebuild prepared data (includes `msg_index`), then run the context-window K screen, late-fuse with the SVM on validation, three-seed the top survivors, and open grouped test only after freeze. Do not regenerate the progress report unless mean test F1 beats the SVM baseline.

In [5]:
# Re-prepare so msg_index / within-match order match the current prepare script.
!python scripts/prepare_l2dtnh.py
!python scripts/run_context_hybrid_experiments.py --device cuda

# Optional: keep the older single-message screen for reference (already frozen as weight_7).
# !python scripts/run_lstm_experiments.py --device cuda --screen-only


Wrote: /content/drive/MyDrive/APS360/Final_Project/data/l2dtnh/l2dtnh_prepared.csv
Audit: /content/drive/MyDrive/APS360/Final_Project/artifacts/data_audit.json
Rows: 14,293
Groups: 101
Class counts:
label
0    12973
1     1320
Split summary:
        rows  positives  groups
split                          
test    2134        195      16
train  10050        927      70
val     2109        198      15
Baseline validation: balanced accuracy 0.826; toxic-class F1 0.629
Run: ctx_k1_weight7_seed42; device: cuda; test enabled: False
Epoch 01/15 | train loss 0.8974 f1@.50 0.312 | val loss 0.7273 p 0.523 r 0.510 f1 0.517 @ 0.75 <- saved
Epoch 02/15 | train loss 0.6071 f1@.50 0.500 | val loss 0.6315 p 0.624 r 0.571 f1 0.596 @ 0.70 <- saved
Epoch 03/15 | train loss 0.4484 f1@.50 0.598 | val loss 0.7906 p 0.642 r 0.606 f1 0.623 @ 0.65 <- saved
Epoch 04/15 | train loss 0.3051 f1@.50 0.700 | val loss 0.9669 p 0.602 r 0.596 f1 0.599 @ 0.50
Epoch 05/15 | train loss 0.2067 f1@.50 0.792 | val loss 1.1729

## 6. Review Frozen Comparison

In [6]:
import json
from pathlib import Path

summary_path = Path('artifacts/context_hybrid_experiment_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Winner:', summary['winner'])
    print('Mean test F1:', summary['test_aggregate']['f1'])
    print('Baseline test F1:', summary['baseline_test']['f1'])
    print('Success against baseline:', summary['error_analysis']['success_against_baseline'])
    print('Progress report update allowed:', summary['protocol']['progress_report_update_allowed'])
else:
    summary = json.loads(Path('artifacts/experiment_summary.json').read_text(encoding='utf-8'))
    print('Frozen winner:', summary['winner'])
    print('Baseline test F1:', summary['baseline_test']['f1'])
    print('LSTM three-seed test F1:', summary['lstm_test_aggregate']['f1'])


Winner: weight7_hybrid_late
Mean test F1: {'mean': 0.6663950274695101, 'std': 0.015118032110439309, 'values': [0.680306905370844, 0.6685714285714286, 0.6503067484662577]}
Baseline test F1: 0.6547884187082406
Success against baseline: True
Progress report update allowed: True


## 7. Verify Canonical Artifacts

The repository already lives in Drive, and every run writes directly to the canonical `artifacts/` directory. No post-run copying is needed.

In [7]:
from pathlib import Path

artifact_dir = Path(PROJECT_DIR) / 'artifacts'
required = [
    'baseline_metrics.json',
    'context_hybrid_metrics.json',
    'context_hybrid_experiment_summary.json',
    'frozen_context_hybrid_config.json',
]
legacy = [
    'lstm_metrics.json',
    'experiment_summary.json',
    'frozen_lstm_config.json',
    'best_model.pt',
]
missing = [name for name in required if not (artifact_dir / name).exists()]
if missing:
    legacy_missing = [name for name in legacy if not (artifact_dir / name).exists()]
    if legacy_missing:
        raise FileNotFoundError(
            f'Missing artifacts. context/hybrid: {missing}; legacy: {legacy_missing}'
        )
    print(
        f'Context/hybrid artifacts not ready yet ({missing}); '
        f'legacy evidence present in {artifact_dir}'
    )
else:
    print(f'Context/hybrid evidence verified in {artifact_dir}')


Context/hybrid evidence verified in /content/drive/MyDrive/APS360/Final_Project/artifacts
